# 01 — Data Cleaning

**Input:** `../data/raw/ema_personality_plus_surveys_merged.csv`
**Output:** `../data/processed/pm_day_clean.csv`

**Description:**
- Load raw merged EMA data
- Detect DSI-SS items (4 items A–D) from column labels
- Build one text document per participant-day (PM window)
- Compute DSI outcomes (any risk, moderate, high acuity)
- Identify crisis items and compute crisis totals
- Save cleaned day-level dataset for downstream notebooks

In [2]:
import os
import re
import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "raw", "ema_personality_plus_surveys_merged.csv")
OUT_PATH = os.path.join("..", "data", "processed", "pm_day_clean.csv")

PID_COL = "expiwell_id_clean"
DT_COL = "ema_dt"
TEXT_COL = "ema_text"
PM_START_HOUR = 15

PM_CRISIS_TOTAL = "dailyPM__Total Score from 5 Questions"

RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [4]:
# =========================
# HELPERS
# =========================
def first_nonnull(x: pd.Series):
    x = x.dropna()
    return x.iloc[0] if len(x) else np.nan


def find_dsi_cols_by_prefix(df: pd.DataFrame, prefix: str):
    """Find DSI-SS items A-D from column names containing item stems."""
    cols = [c for c in df.columns if c.startswith(prefix)]

    def pick(patterns):
        pats = [re.compile(p, re.IGNORECASE) for p in patterns]
        hits = [c for c in cols if any(p.search(c) for p in pats)]
        hits = sorted(hits, key=len)
        return hits[0] if hits else None

    A = pick([r"thoughts of killing myself", r"\bkilling myself\b"])
    B = pick([r"definite plan", r"formulated.*plan", r"considered possible ways", r"\bplans?\b"])
    C = pick([r"little or no control", r"control over", r"under my control", r"\bcontrol\b"])
    D = pick([r"impulses to kill myself", r"\bimpulses\b"])

    return {"A": A, "B": B, "C": C, "D": D}


def find_crisis_item_cols(df: pd.DataFrame, prefix: str, total_col: str, top_k: int = 5):
    """Find the top_k item columns most correlated with the total score."""
    cols = [
        c for c in df.columns
        if c.startswith(prefix) and pd.api.types.is_numeric_dtype(df[c])
    ]
    candidates = []
    for c in cols:
        if c == total_col:
            continue
        s = df[c].dropna()
        if len(s) < 100:
            continue
        mn, mx = s.min(), s.max()
        if mn >= 0 and mx <= 4 and s.nunique() == 5:
            tmp = df[[c, total_col]].dropna()
            if len(tmp) < 200:
                continue
            corr = tmp.corr().iloc[0, 1]
            candidates.append((c, float(corr)))

    candidates = sorted(candidates, key=lambda x: -abs(x[1]))
    return [c for c, _ in candidates[:top_k]]

In [6]:
# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH)

# Fallbacks for column names
if PID_COL not in df.columns:
    PID_COL = "expiwell_id"
if DT_COL not in df.columns:
    DT_COL = "start_date"

df[DT_COL] = pd.to_datetime(df[DT_COL], errors="coerce")
df["ema_date"] = df[DT_COL].dt.date
df["hour"] = df[DT_COL].dt.hour

print("Loaded rows:", len(df), "| participants:", df[PID_COL].nunique())

Loaded rows: 2742 | participants: 123


In [8]:
# =========================
# DETECT DSI-SS ITEMS (dailyPM__)
# =========================
DSI_PM = find_dsi_cols_by_prefix(df, "dailyPM__")
print("Detected dailyPM DSI columns:", DSI_PM)

if any(v is None for v in DSI_PM.values()):
    raise ValueError("Could not detect all 4 dailyPM__ DSI items. Adjust patterns.")

dsi_cols = [DSI_PM[k] for k in ["A", "B", "C", "D"]]
df["dsi_PM_total"] = df[dsi_cols].sum(axis=1, min_count=4)
df["dsi_PM_max"] = df[dsi_cols].max(axis=1)

Detected dailyPM DSI columns: {'A': "dailyPM__[ 0 = 'I am not having thoughts of killing myself. ' 1 = 'Sometimes I have had thoughts of killing myself.' 2 = 'Most of the time I have had thoughts of killing myself. ' 3 = 'I am always having thoughts of killing myself.'  ]", 'B': "dailyPM__[ 0 = 'I have not had any thoughts about suicide. ' 1 = 'I have had thoughts about suicide but not formulated any plans. ' 2 = 'I have been having thoughts of suicide and have considered possible ways of doing it. ' 3 = 'I am having thoughts about suicide and have formulated a definite plan.'  ]", 'C': "dailyPM__[ 0 = 'I have not had thoughts about suicide. ' 1 = 'I have been having thoughts about suicide but have these thoughts completely under my control. ' 2 = 'I have been having thoughts about suicide but have these thoughts somewhat under my control. ' 3 = 'I have been having thoughts about suicide but have little or no control over these thoughts. '  ]", 'D': "dailyPM__[ 0 = 'I have not been hav

In [10]:
# =========================
# IDENTIFY CRISIS ITEMS (to exclude from baseline covariates)
# =========================
if PM_CRISIS_TOTAL not in df.columns:
    raise ValueError(f"Missing PM crisis total column: {PM_CRISIS_TOTAL}")

CRISIS_PM_ITEM_COLS = find_crisis_item_cols(df, "dailyPM__", PM_CRISIS_TOTAL, top_k=5)
print("Crisis PM item columns identified:")
for c in CRISIS_PM_ITEM_COLS:
    print(" ", c)

# Build list of OTHER PM numeric covariates (excluding crisis items and DSI items)
dailyPM_numeric = [
    c for c in df.columns
    if c.startswith("dailyPM__") and pd.api.types.is_numeric_dtype(df[c])
]
EXCLUDE = set([PM_CRISIS_TOTAL] + CRISIS_PM_ITEM_COLS + dsi_cols)
OTHER_PM_COLS = [c for c in dailyPM_numeric if c not in EXCLUDE]

print(f"Other PM numeric covariates: {len(OTHER_PM_COLS)}")

Crisis PM item columns identified:
  dailyPM__[ 0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely'0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely'0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely' ]
  dailyPM__[ 0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely' ]
  dailyPM__[ 0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely'0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely' ]
  dailyPM__[ 0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely'0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely'0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely'0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely' ]
  dailyPM__[ 0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Quite a bit'4 = 'Extremely'0 = 'Not at all'1 = 'A little'2 = 'Somewhat'3 = 'Qui

In [12]:
# =========================
# BUILD PM-WINDOW TEXT PER PERSON-DAY
# =========================
pm_text_df = df.loc[df["hour"].notna() & (df["hour"] >= PM_START_HOUR)].copy()

pm_text_day = (
    pm_text_df.sort_values([PID_COL, DT_COL])
    .groupby([PID_COL, "ema_date"], as_index=False)
    .agg(
        pm_day_text=(TEXT_COL, lambda s: " ".join([str(t) for t in s.dropna()])),
        n_pm_text=(TEXT_COL, lambda s: int(s.notna().sum()))
    )
)

In [14]:
# =========================
# BUILD DAY-LEVEL TABLE
# =========================
agg_dict = {
    "dsi_PM_total": ("dsi_PM_total", first_nonnull),
    "dsi_PM_max": ("dsi_PM_max", first_nonnull),
    "crisis_PM": (PM_CRISIS_TOTAL, first_nonnull),
}
for c in OTHER_PM_COLS:
    agg_dict[c] = (c, first_nonnull)

day = (
    df.sort_values([PID_COL, DT_COL])
    .groupby([PID_COL, "ema_date"], as_index=False)
    .agg(**agg_dict)
)

day = day.merge(pm_text_day, on=[PID_COL, "ema_date"], how="left")
day["pm_day_text"] = day["pm_day_text"].fillna("").astype(str)

# Keep usable rows: has text + DSI + crisis
m = day["pm_day_text"].str.strip().ne("") & day["dsi_PM_total"].notna() & day["crisis_PM"].notna()
day = day.loc[m].reset_index(drop=True)

# Create binary DSI outcomes
day["any_risk_total_gt0"] = (day["dsi_PM_total"] > 0).astype(int)
day["moderate_total_ge2"] = (day["dsi_PM_total"] >= 2).astype(int)
day["high_any_item_eq3"] = (day["dsi_PM_max"] >= 3).astype(int)

# Rename crisis column for clarity
day.rename(columns={"crisis_PM": "crisis_PM_from_full"}, inplace=True)

print("Usable rows:", len(day), "| participants:", day[PID_COL].nunique())
print("Pos rates:")
print("  any_risk:", round(day["any_risk_total_gt0"].mean(), 4))
print("  moderate:", round(day["moderate_total_ge2"].mean(), 4))
print("  high_acuity:", round(day["high_any_item_eq3"].mean(), 4))

Usable rows: 2511 | participants: 123
Pos rates:
  any_risk: 0.3744
  moderate: 0.3493
  high_acuity: 0.0394


In [15]:
# =========================
# SAVE
# =========================
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
day.to_csv(OUT_PATH, index=False)
print("Saved cleaned dataset:", OUT_PATH)
print("Shape:", day.shape)

Saved cleaned dataset: ..\data\processed\pm_day_clean.csv
Shape: (2511, 26)
